In [1]:
import os
import pickle
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

# Paths relative to RFModel/
TRAIN_PATH = "mqttset/Data/FINAL_CSV/train70.csv"
TEST_PATH = "mqttset/Data/FINAL_CSV/test30.csv"
TARGET_COL = "target"
BENIGN_LABEL = "legitimate"

# Paper-inspired feature set
CANDIDATE_FEATURES = [
    "mqtt.len",
    "tcp.time_delta",
    "mqtt.dupflag",
    "mqtt.conflag.cleansess",
    "mqtt.kalive",
]

if not os.path.exists(TRAIN_PATH):
    raise FileNotFoundError(f"Missing training file: {TRAIN_PATH}")
if not os.path.exists(TEST_PATH):
    raise FileNotFoundError(f"Missing test file: {TEST_PATH}")

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

if TARGET_COL not in train_df.columns or TARGET_COL not in test_df.columns:
    raise ValueError(f"Expected label column '{TARGET_COL}' in both train and test files")

train_df[TARGET_COL] = train_df[TARGET_COL].astype(str).str.strip().str.lower()
test_df[TARGET_COL] = test_df[TARGET_COL].astype(str).str.strip().str.lower()

# 0 = benign, 1 = attacker (all non-legitimate labels)
y_train = (train_df[TARGET_COL] != BENIGN_LABEL).astype(int).values
y_test = (test_df[TARGET_COL] != BENIGN_LABEL).astype(int).values

FEATURES = [c for c in CANDIDATE_FEATURES if c in train_df.columns and c in test_df.columns]
if not FEATURES:
    raise ValueError("No candidate features found in both train and test files")

X_train_df = train_df[FEATURES].apply(pd.to_numeric, errors="coerce")
X_test_df = test_df[FEATURES].apply(pd.to_numeric, errors="coerce")

# Keep only rows where selected features are valid numbers
train_mask = X_train_df.notna().all(axis=1)
test_mask = X_test_df.notna().all(axis=1)

X_train = X_train_df.loc[train_mask].values
y_train = y_train[train_mask.values]
X_test = X_test_df.loc[test_mask].values
y_test = y_test[test_mask.values]

print("Features used:", FEATURES)
print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")
print(f"Train class counts -> attacker: {int(y_train.sum())}, benign: {int((y_train == 0).sum())}")
print(f"Test class counts  -> attacker: {int(y_test.sum())}, benign: {int((y_test == 0).sum())}")

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample",
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["benign", "attacker"]))

if len(np.unique(y_test)) == 2:
    y_prob = rf.predict_proba(X_test)[:, 1]
    print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

# Save plain model for compatibility with existing scripts
with open("rf_model.pkl", "wb") as f:
    pickle.dump(rf, f)

# Save richer artifact with feature order and class mapping
artifact = {
    "model": rf,
    "features": FEATURES,
    "target_mapping": {"benign": 0, "attacker": 1},
    "benign_label": BENIGN_LABEL,
}
with open("rf_attack_benign.pkl", "wb") as f:
    pickle.dump(artifact, f)

with open("rf_features.txt", "w") as f:
    f.write("\n".join(FEATURES))

print("\nModel saved to rf_model.pkl and rf_attack_benign.pkl")
print("Feature list saved to rf_features.txt")

/var/folders/lm/x7r91bx97yx5fcw08yw2bwy80000gn/T/ipykernel_57518/2928319989.py:29: DtypeWarning: Columns (3,14,19,23) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(TRAIN_PATH)
/var/folders/lm/x7r91bx97yx5fcw08yw2bwy80000gn/T/ipykernel_57518/2928319989.py:30: DtypeWarning: Columns (3,14,19,23) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df = pd.read_csv(TEST_PATH)


Features used: ['mqtt.len', 'tcp.time_delta', 'mqtt.dupflag', 'mqtt.conflag.cleansess', 'mqtt.kalive']
Train size: 8456823 | Test size: 3624366
Train class counts -> attacker: 115822, benign: 8341001
Test class counts  -> attacker: 49651, benign: 3574715

Test Accuracy: 97.15%

Confusion Matrix:
[[3475446   99269]
 [   4080   45571]]

Classification Report:
              precision    recall  f1-score   support

      benign       1.00      0.97      0.99   3574715
    attacker       0.31      0.92      0.47     49651

    accuracy                           0.97   3624366
   macro avg       0.66      0.95      0.73   3624366
weighted avg       0.99      0.97      0.98   3624366

ROC-AUC: 0.9853

Model saved to rf_model.pkl and rf_attack_benign.pkl
Feature list saved to rf_features.txt


In [4]:
import os
import pickle
import numpy as np
import pandas as pd

ARTIFACT_PATH = "rf_attack_benign.pkl"
INFER_INPUT_PATH = "mqttset/Data/FINAL_CSV/test30.csv"
N_DEMO_ROWS = 10
THRESHOLD = 0.50

if not os.path.exists(ARTIFACT_PATH):
    raise FileNotFoundError(
        f"Missing model artifact '{ARTIFACT_PATH}'. Run Cell 1 first."
    )

with open(ARTIFACT_PATH, "rb") as f:
    artifact = pickle.load(f)

model = artifact["model"]
features = artifact["features"]
print("Loaded model artifact:", ARTIFACT_PATH)
print("Feature order:", features)


def predict_attack_benign(df, model, features, threshold=0.5):
    x = df.reindex(columns=features).apply(pd.to_numeric, errors="coerce")
    valid_mask = x.notna().all(axis=1)

    dropped = int((~valid_mask).sum())
    if dropped > 0:
        print(f"Dropped {dropped} rows with missing/non-numeric feature values")

    x_valid = x.loc[valid_mask]
    if x_valid.empty:
        return pd.DataFrame(columns=list(df.columns) + [
            "attack_probability",
            "pred_class_id",
            "pred_class_name",
        ])

    x_values = x_valid.values
    if hasattr(model, "predict_proba"):
        attack_prob = model.predict_proba(x_values)[:, 1]
        pred_id = (attack_prob >= threshold).astype(int)
    else:
        pred_id = model.predict(x_values)
        attack_prob = np.full(shape=len(pred_id), fill_value=np.nan)

    out = df.loc[x_valid.index].copy()
    out["attack_probability"] = attack_prob
    out["pred_class_id"] = pred_id
    out["pred_class_name"] = np.where(pred_id == 1, "attacker", "benign")
    return out


if not os.path.exists(INFER_INPUT_PATH):
    raise FileNotFoundError(f"Missing inference input file: {INFER_INPUT_PATH}")

demo_df = pd.read_csv(INFER_INPUT_PATH, nrows=N_DEMO_ROWS, low_memory=False)
demo_pred = predict_attack_benign(demo_df, model, features, threshold=THRESHOLD)

print(f"\nDemo predictions on first {N_DEMO_ROWS} rows from {INFER_INPUT_PATH}:")
show_cols = features + ["attack_probability", "pred_class_name"]
print(demo_pred[show_cols].to_string(index=False))

Loaded model artifact: rf_attack_benign.pkl
Feature order: ['mqtt.len', 'tcp.time_delta', 'mqtt.dupflag', 'mqtt.conflag.cleansess', 'mqtt.kalive']

Demo predictions on first 10 rows from mqttset/Data/FINAL_CSV/test30.csv:
 mqtt.len  tcp.time_delta  mqtt.dupflag  mqtt.conflag.cleansess  mqtt.kalive  attack_probability pred_class_name
      8.0        0.998772           0.0                     0.0          0.0            0.000000          benign
     11.0        1.000038           0.0                     0.0          0.0            0.000000          benign
      8.0        0.000157           0.0                     0.0          0.0            0.000000          benign
     11.0        0.000092           0.0                     0.0          0.0            0.000000          benign
      0.0        0.000075           0.0                     0.0          0.0            0.875067        attacker
     11.0        0.999883           0.0                     0.0          0.0            0.000000    